<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
DecisionTreeRegressor - Diamonds - pmml
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install
!uv pip install --system -q nyoka pypmml

In [ ]:
# Daten & Strukturen
from pandas import read_csv  # CSV laden

# Datenvorverarbeitung
from sklearn.preprocessing import Ordinalcoder  # Encoding

# Pipeline
from sklearn.pipeline import Pipeline  # Verarbeitungsschritte kombinieren

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell
from sklearn.tree import DecisionTreeRegressor  # Regressionsmodell

# Evaluation & Metriken
from sklearn.metrics import r2_score, mean_absolute_error  # Regressionsmetriken

# Modellbereitstellung & PMML
from nyoka import skl_to_pmml  # Modell nach PMML exportieren
from pypmml import Model  # PMML-Modelle laden und ausführen

In [ ]:
# System & Konfiguration
import warnings  # Warnungen steuern
warnings.filterwarnings("ignore")  # Warnungen unterdrücken

# 1 | Understand
***

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>
✅ Passenden Transfer-Case als Orientierung auswählen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Duplikate ermitteln/löschen</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>
✅ Preprocessing-Entscheidungen aus dem Transfer-Case auf Übertragbarkeit prüfen</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Split
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

coder = Ordinalcoder(categories=cat_seq)
coder.fit(data_train[cat_col])
data_train[cat_col] = coder.transform(data_train[cat_col])
data_test[cat_col] = coder.transform(data_test[cat_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Modellansatz aus dem Transfer-Case als Orientierung für die eigene Lösung nutzen</br>

<p><font color='black' size="5">🌳 Algorithmus: Entscheidungsbaum (Decision Tree – Regression)</font></p>

Ein Entscheidungsbaum für Regression funktioniert wie der Klassifikationsbaum – statt einer Klasse sagt er am Ende jedoch einen kontinuierlichen Zahlenwert vorher (z.B. einen Preis). In den Blättern steht der Mittelwert der Trainingsdaten, die in dieses Blatt fallen.

**Analogie:** Wie ein Sachverständiger, der den Wert einer Immobilie anhand von Merkmalen (Größe, Lage, Baujahr) schrittweise einschränkt.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `max_depth` | Maximale Tiefe (begrenzt Overfitting) |
| `criterion` | Gütekriterium: `squared_error` oder `absolute_error` |
| `random_state` | Reproduzierbarkeit |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum – Regression</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    ROOT[Wurzel: Karat > 1.0?] -->|ja| A[Klarheit: SI2 oder schlechter?]
    ROOT -->|nein| B[Karat > 0.5?]

    A -->|ja| C[Ø Preis: $5.400]
    A -->|nein| D[Ø Preis: $9.200]

    B -->|ja| E[Ø Preis: $2.100]
    B -->|nein| F[Ø Preis: $750]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class ROOT input;
    class C,D,E,F output;
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=500))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = DecisionTreeRegressor(max_depth=5, random_state=42)

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Cross-Validation</br>
✅ Bootstrapping</br>
✅ Regularisierung prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>
✅ Ergebnisse mit dem Transfer-Case vergleichen und Unterschiede erklären</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"Modell: {model} -- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"Modell: {model} -- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"Modell: {model} -- Test -- Mean Absolute Error: {mae:5.2f}")

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>
✅ Grenzen der Übertragbarkeit vom Transfer-Case auf den eigenen Anwendungsfall dokumentieren</br>

| Modul/Bibliothek | Einzelnes Modell speichern/laden | Ganze Pipeline speichern/laden | Typischer Einsatz | Hinweis |
|---|---:|---:|---|---|
| `pickle` | Ja | Ja, wenn die Pipeline picklable ist | Einfache Python-Serialisierung | Schnell und direkt, aber nur für vertrauenswürdige Dateien geeignet. |
| `joblib` | Ja | Ja | scikit-learn-Modelle mit großen NumPy-Arrays | Für scikit-learn sehr üblich und oft die beste Standardwahl. |
| `skops.io` | Ja | Ja, sofern unterstützt | Sichereres Modell-Persistieren | Eher geeignet, wenn Sicherheit wichtiger ist als maximale Einfachheit. |
| `ONNX` | Ja, nach Konvertierung | Eher nicht nativ | Deployment außerhalb von Python | Gut für Portabilität, aber nicht die direkteste scikit-learn-Speicherung. |
| `PMML` | Ja, nach Konvertierung | Eher nicht nativ | Austauschformat / Langzeitarchivierung | Praktisch für Interoperabilität, aber meist mit zusätzlichem Konvertierungsschritt. |

Für einen normalen scikit-learn-Workflow ist `joblib` meist die beste Wahl, besonders man eine komplette `Pipeline` speichern möchte. Wenn mN nur ein einzelnes Modell sichern möchtest, funktionieren `PMML`, `pickle` und `joblib` ebenfalls gut.

<p><font size="5">
Speichern als PMML
</p>

In [ ]:
# save the model to PMML
pipeline_ = Pipeline([("estimator", model)])
skl_to_pmml(
    pipeline=pipeline_,
    col_names=data_test.columns,
    pmml_f_name="/content/diamonds_model.pmml",
)

<p><font size="5">
Laden
</p>

In [ ]:
model_geladen = Model.fromFile("/content/diamonds_model.pmml")

<p><font size="5">
Prognose
</p>

In [ ]:
target_test_pred_new = model_geladen.predict(data_test)